# Lezione 9 — Dashboard con Dash: fondamenti

## Perché questo ti riguarda?

Finora hai generato grafici: li esegui, li guardi, magari li salvi come immagine. Questo funziona benissimo per un report chiuso. Ma nel momento in cui qualcuno ti dice "e se filtro per anno?" o "posso vederlo io stesso?", un grafico statico non basta più. Dovresti tornare al tuo script, cambiare un valore, eseguirlo di nuovo e inviare un'altra immagine.

Una dashboard dà a quella persona il controllo. Apre una pagina nel browser, muove un controllo, e il grafico si aggiorna da solo — senza che tu intervenga. **Dash** è lo strumento che ti permette di costruire questo usando solo Python, senza dover imparare a fondo HTML, CSS o JavaScript.

## ⚠️ Nota tecnica importante: Dash e i notebook

Dash **non è una libreria di grafici che si mostra in una cella** — è un micro-framework che avvia un **server web** e serve la tua applicazione a un URL (normalmente `http://127.0.0.1:8050/`). Questo significa che, a differenza di Matplotlib o Plotly, **non puoi semplicemente eseguire una cella e vedere il risultato inline** senza altro.

Hai due opzioni per lavorare con Dash da questo materiale:

**Opzione A — Il modo "ufficiale" del corso (consigliato):** salva il codice in un file `.py` (ad esempio `dash_basics.py`) ed eseguilo da un terminale, sia sulla tua macchina locale sia nell'IDE nel cloud di Skills Network (Theia). È esattamente come fa il materiale originale di IBM:

```
python dash_basics.py
```

Il terminale mostrerà qualcosa come `Dash is running on http://127.0.0.1:8050/`. Apri quell'indirizzo nel browser e lì vedi la tua dashboard.

**Opzione B — Eseguirlo dentro questo stesso notebook (Colab o Jupyter locale):** Dash normale blocca la cella aspettando connessioni se lo avvii con `app.run()`. Le versioni recenti di Dash (2.11 in poi, che è ciò che installa `pip install dash` oggi) portano supporto nativo per i notebook: basta chiamare `app.run(jupyter_mode="inline")` invece di `app.run()`, senza installare pacchetti aggiuntivi né cambiare `dash.Dash` con nulla di diverso.

Più sotto trovi il codice della dashboard scritto in **entrambi i modi**: una cella di codice con il contenuto esatto che copieresti in un file `.py` (Opzione A), e una cella opzionale che lo esegue qui stesso con `jupyter_mode="inline"` (Opzione B). Usa quella che ti risulta più comoda — il contenuto della dashboard è identico in entrambe.

## L'analogia: il cruscotto di un'auto

Pensa a Dash come al cruscotto di un'auto. Tu, come progettista, decidi quali strumenti ci sono: il tachimetro, la radio, le spie di avviso. Il guidatore non deve sapere come funziona il motore internamente — interagisce solo con i controlli che hai messo lì.

Costruire una dashboard con Dash è esattamente questo: decidi tu quali "schermi" (grafici) e quali "pulsanti" (controlli) avrà la tua applicazione, e quali informazioni mostra ogni schermo. In questa lezione costruirai il cruscotto più semplice possibile: solo schermi, senza pulsanti ancora. I pulsanti — l'interattività vera con `@app.callback` — arrivano nella prossima lezione.

## I tre blocchi di ogni app Dash

Ogni applicazione Dash, per quanto semplice o complessa sia, si costruisce con tre blocchi che appaiono sempre nello stesso ordine:

1. **Import**: porti le librerie che userai (Dash, i suoi componenti, e normalmente pandas e Plotly per preparare i dati e le figure).
2. **`app.layout`**: definisci la struttura visiva della pagina — cosa appare e in che ordine. Qui è dove usi `html.Div`, `html.H1`, `html.P` e `dcc.Graph`.
3. **`app.run()`**: avvii un server web locale che serve la tua applicazione nel browser.

Vediamo ogni blocco con l'esempio reale di una dashboard di compagnie aeree: un pie chart che mostra la proporzione di voli per gruppo di distanza.

### Blocco 1: gli import

```python
import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html
```

Niente di nuovo qui se hai già usato pandas e Plotly Express: `pd` per gestire i dati, `px` per creare le figure. La novità è `dash` e i suoi due sottomoduli:

- `dcc` (**D**ash **C**ore **C**omponents): componenti "intelligenti" come grafici, menu a tendina o campi di input.
- `html`: componenti che riflettono i tag HTML di base — titoli, paragrafi, divisioni — ma scritti come funzioni Python.

### Preparare i dati e la figura

```python
airline_data = pd.read_csv(
    'https://.../airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

data = airline_data.sample(n=500, random_state=42)

fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')
```

Nota che la figura (`fig`) viene creata con Plotly Express, esattamente come nelle lezioni precedenti. **Dash non sostituisce Plotly — lo usa.** Dash si occupa di "metterlo in una pagina web e interagire con l'utente"; Plotly continua a essere ciò che disegna il grafico.

### Blocco 2: `app.layout`

```python
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])
```

Riga per riga:

- `app = dash.Dash(__name__)` crea l'applicazione. È la riga che va sempre prima del layout.
- `app.layout = html.Div(children=[...])` definisce **tutto** ciò che si vede nella pagina. `html.Div` è un contenitore — come una scatola che raggruppa elementi — e riceve una lista (`children`) con i componenti che mostrerà, in ordine.
- `html.H1(...)` è il titolo principale. Il parametro `style` è un dizionario CSS: qui lo centri (`textAlign`), gli dai un colore specifico (`color`) e una dimensione del testo (`font-size`).
- `html.P(...)` è un paragrafo di testo — la descrizione del grafico, anch'essa centrata e con il proprio colore.
- `dcc.Graph(figure=fig)` è il componente che disegna effettivamente il grafico. Gli passi la figura Plotly che hai creato prima con il parametro `figure`.

Nota il parallelismo: ogni elemento visivo della tua dashboard (titolo, descrizione, grafico) corrisponde a un elemento dentro quella lista `children`. Se vuoi aggiungere qualcos'altro alla pagina, aggiungi un altro elemento alla lista.

### Blocco 3: `app.run()`

```python
if __name__ == '__main__':
    app.run()
```

Questo avvia il server. Non genera un'immagine né apre una finestra da solo — lascia in esecuzione un processo che aspetta connessioni dal browser.

## Codice completo — Opzione A: versione da salvare come `.py` ed eseguire in terminale

Questa è la versione "canonica": copia il contenuto della cella seguente in un file, ad esempio `dash_basics.py`, e in un terminale (locale o nell'IDE di Skills Network) esegui:

```
python dash_basics.py
```

Il terminale mostrerà qualcosa come `Dash is running on http://127.0.0.1:8050/`. Apri quell'indirizzo nel tuo browser per vedere il titolo, la descrizione e il pie chart. Finché il processo continua a girare nel terminale, la pagina resta disponibile; per fermarlo usa `Ctrl+C`.

Eseguire questa cella dentro il notebook **non produrrà nulla di utile** (o potrebbe addirittura restare "bloccata" aspettando connessioni) perché `dash.Dash` è pensato per processi indipendenti, non per celle di Jupyter — per questo più sotto trovi l'Opzione B.

In [ ]:
# ============================================================
# VERSIONE DA SALVARE COME dash_basics.py ED ESEGUIRE IN TERMINALE
# (python dash_basics.py)
# ============================================================

# Blocco 1: import
import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html

# Leggere i dati delle compagnie aeree in un DataFrame di pandas
airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

# Campione casuale di 500 punti. random_state fisso per riproducibilità.
data = airline_data.sample(n=500, random_state=42)

# Creazione del pie chart con Plotly Express
fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')

# Blocco 2: app.layout
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])

# Blocco 3: app.run()
if __name__ == '__main__':
    app.run()


## Codice completo — Opzione B (opzionale): eseguirlo qui stesso, senza uscire dal notebook

Se vuoi vedere la dashboard **senza uscire dal notebook** (ad esempio in Google Colab), usa esattamente lo stesso codice dell'Opzione A, cambiando solo l'ultima riga: `app.run(jupyter_mode="inline")` invece di `app.run()`. Non serve installare nulla di aggiuntivo — questa capacità è già inclusa in `dash` dalla versione 2.11.

Questa cella è **opzionale**: eseguila solo se vuoi davvero vedere il risultato renderizzato dentro questo notebook.

In [ ]:
# ============================================================
# VERSIONE OPZIONALE DA ESEGUIRE DENTRO IL NOTEBOOK (senza dipendenze extra)
# ============================================================

import pandas as pd
import plotly.express as px
import dash
from dash import dcc
from dash import html

airline_data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv',
    encoding="ISO-8859-1",
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str})

data = airline_data.sample(n=500, random_state=42)

fig = px.pie(data, values='Flights', names='DistanceGroup',
             title='Distance group proportion by flights')

app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1('Airline Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),
    html.P('Proportion of distance group (250 mile distance interval group) by flights.',
           style={'textAlign': 'center', 'color': '#F57241'}),
    dcc.Graph(figure=fig),
])

# jupyter_mode="inline" renderizza l'app dentro l'output di questa cella
app.run(jupyter_mode="inline")

## Punti chiave

- Ogni app Dash ha tre blocchi: **import**, **`app.layout`** e **`app.run()`**.
- `app.layout` riceve un `html.Div` con una lista `children` che definisce, in ordine, tutto ciò che appare nella pagina.
- `html.H1` e `html.P` sono testo (titolo e paragrafo); `dcc.Graph` è il componente che mostra una figura Plotly tramite il suo parametro `figure`.
- Il parametro `style` nei componenti HTML è un dizionario CSS (`textAlign`, `color`, `font-size`, ecc.) che controlla il loro aspetto.
- Dash non sostituisce Plotly: le figure si continuano a creare con `plotly.express` o `plotly.graph_objects`; Dash le colloca solo dentro una pagina web interattiva.
- In un ambiente normale, esegui l'app con `python file.py` e la apri nel browser usando la porta indicata dal terminale (normalmente `8050`). Dentro un notebook, usa `app.run(jupyter_mode="inline")` invece di `app.run()` — Dash 2.11+ lo include già, senza installare nulla di aggiuntivo.
- I pulsanti e i controlli interattivi (`@app.callback`) arrivano nella prossima lezione — oggi abbiamo costruito solo "schermi".